In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("Automobile_data.csv")

df

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,95,volvo,gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845
201,-1,95,volvo,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045
202,-1,95,volvo,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485
203,-1,95,volvo,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.4,23.0,106,4800,26,27,22470


In [7]:
# Replace ? with NaN
df = df.replace('?', np.nan)

print(df.head())

   symboling normalized-losses         make fuel-type aspiration num-of-doors  \
0          3               NaN  alfa-romero       gas        std          two   
1          3               NaN  alfa-romero       gas        std          two   
2          1               NaN  alfa-romero       gas        std          two   
3          2               164         audi       gas        std         four   
4          2               164         audi       gas        std         four   

    body-style drive-wheels engine-location  wheel-base  ...  engine-size  \
0  convertible          rwd           front        88.6  ...          130   
1  convertible          rwd           front        88.6  ...          130   
2    hatchback          rwd           front        94.5  ...          152   
3        sedan          fwd           front        99.8  ...          109   
4        sedan          4wd           front        99.4  ...          136   

   fuel-system  bore  stroke compression-ratio hor

In [9]:
cols = ['horsepower', 'peak-rpm', 'price', 'bore', 'stroke']

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[cols].dtypes)

horsepower    float64
peak-rpm      float64
price         float64
bore          float64
stroke        float64
dtype: object


In [11]:
for col in cols:
    df[col] = df.groupby('body-style')[col].transform(
        lambda x: x.fillna(x.median())
    )

print(df[cols].isnull().sum())

horsepower    0
peak-rpm      0
price         0
bore          0
stroke        0
dtype: int64


In [13]:
df['power_to_weight_ratio'] = df['horsepower'] / df['curb-weight']

print(df[['horsepower', 'curb-weight', 'power_to_weight_ratio']].head())

   horsepower  curb-weight  power_to_weight_ratio
0       111.0         2548               0.043564
1       111.0         2548               0.043564
2       154.0         2823               0.054552
3       102.0         2337               0.043646
4       115.0         2824               0.040722


In [15]:
df['combined_mpg'] = (df['city-mpg'] + df['highway-mpg']) / 2

print(df[['city-mpg', 'highway-mpg', 'combined_mpg']].head())

   city-mpg  highway-mpg  combined_mpg
0        21           27          24.0
1        21           27          24.0
2        19           26          22.5
3        24           30          27.0
4        18           22          20.0


In [17]:
body_summary = df.groupby('body-style')[
    ['horsepower', 'price', 'bore', 'stroke',
     'power_to_weight_ratio', 'combined_mpg']
].agg(['count', 'mean', 'median', 'min', 'max'])

print(body_summary)

            horsepower                                 price                \
                 count        mean median   min    max count          mean   
body-style                                                                   
convertible          6  131.666667  113.5  90.0  207.0     6  21890.500000   
hardtop              8  142.250000  119.5  69.0  207.0     8  22208.500000   
hatchback           70  101.142857   88.0  48.0  288.0    70   9920.714286   
sedan               96  103.104167   96.0  52.0  262.0    96  14389.312500   
wagon               25   97.620000   94.5  62.0  162.0    25  12371.960000   

                                        ... power_to_weight_ratio            \
              median      min      max  ...                 count      mean   
body-style                              ...                                   
convertible  17084.5  11595.0  37028.0  ...                     6  0.047006   
hardtop      19687.5   8249.0  45400.0  ...                

In [19]:
engine_summary = df.groupby('engine-location')[
    ['horsepower', 'price', 'power_to_weight_ratio', 'combined_mpg']
].agg(['count', 'mean', 'median', 'min', 'max'])

print(engine_summary)

                horsepower                                  price  \
                     count        mean median    min    max count   
engine-location                                                     
front                  202  102.601485   95.0   48.0  288.0   202   
rear                     3  207.000000  207.0  207.0  207.0     3   

                                                        power_to_weight_ratio  \
                        mean   median      min      max                 count   
engine-location                                                                 
front            12824.50495  10221.5   5118.0  45400.0                   202   
rear             34528.00000  34028.0  32528.0  37028.0                     3   

                                                        combined_mpg  \
                     mean    median       min       max        count   
engine-location                                                        
front            0.039553  0.037